# Phase 1: Prompt Asset Layer Plan

This notebook narrows the roadmap to the prompt asset layer.

It is meant to answer:
- what already exists,
- what the prompt asset layer should own,
- what the manifest and resolver contract should look like,
- how to migrate from inline messages to explicit prompt refs without hidden overrides.

Primary design inputs:
- `llm_client/docs/adr/0011-prompt-assets-explicit-identity.md`
- `llm_client/docs/ECOSYSTEM_TOP_DOWN_ARCHITECTURE.md`
- `prompt_eval/docs/UNCERTAINTIES.md` (especially U3)


## How To Read This

- The early cells are executable and show current behavior.
- The middle cells define the target manifest and resolver shape.
- The later cells are planning cells for the migration phases.
- Some cells are deliberately pseudocode-like because the supporting package does not exist yet.

The point is not to overdesign. The point is to make the Phase 1 decisions explicit enough that an agent can implement them in thin verified slices.


In [ ]:
import sys
import tempfile
from pathlib import Path
from pprint import pprint

PROJECTS = Path.home() / "projects"
llm_client_repo = PROJECTS / "llm_client"
repo_str = str(llm_client_repo)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)


In [1]:
prompt_phase_state = {
    "already_done": [
        "prompts treated as YAML/Jinja data",
        "llm_client.render_prompt(path, **context) exists",
        "prompt_eval PromptVariant supports prompt_ref metadata",
        "observability records prompt_ref when supplied",
    ],
    "current_gap": [
        "no shared prompt registry or asset resolver yet",
        "projects still mostly own prompt files locally",
        "inline PromptVariant.messages remain the compatibility path",
    ],
    "phase_1_goal": "introduce explicit prompt asset identity and lineage without hidden override resolution",
}
pprint(prompt_phase_state)


{'already_done': ['prompts treated as YAML/Jinja data',
                   'llm_client.render_prompt(path, **context) exists',
                   'prompt_eval PromptVariant supports prompt_ref metadata',
                   'observability records prompt_ref when supplied'],
 'current_gap': ['no shared prompt registry or asset resolver yet',
                 'projects still mostly own prompt files locally',
                 'inline PromptVariant.messages remain the compatibility path'],
 'phase_1_goal': 'introduce explicit prompt asset identity and lineage without hidden override resolution'}


## Executable Demo: Current `render_prompt()` Behavior

This cell proves the current baseline: prompt templates already live as YAML/Jinja data and render deterministically from an explicit file path.


In [2]:
from llm_client import render_prompt

root = Path(tempfile.mkdtemp(prefix="prompt_asset_notebook_"))
prompt_path = root / "prompts" / "demo.yaml"
prompt_path.parent.mkdir(parents=True, exist_ok=True)
prompt_path.write_text(
    (
        "name: demo\n"
        "version: \"1.0\"\n"
        "messages:\n"
        "  - role: system\n"
        "    content: |\n"
        "      You are a concise analyst.\n"
        "  - role: user\n"
        "    content: |\n"
        "      Summarize {{ topic }} in {{ style }} style.\n"
    ),
    encoding="utf-8",
)

messages = render_prompt(prompt_path, topic="prompt assets", style="bullet")
pprint(messages)


[{'content': 'You are a concise analyst.', 'role': 'system'},
 {'content': 'Summarize prompt assets in bullet style.', 'role': 'user'}]


## Design Constraints For Phase 1

Non-negotiable constraints from the ADRs and discussion:

- prompts remain data,
- shared prompt assets are not owned by `prompt_eval`,
- no hidden project-local override mechanism as the default,
- customization creates a new asset with lineage,
- `llm_client.render_prompt()` should eventually resolve prompt refs deterministically,
- `prompt_eval` should evaluate prompt assets and record their IDs, not become the prompt registry.


In [3]:
prompt_asset_manifest = {
    "ref": "shared.summarize.concise@1",
    "owner": "shared",
    "task_family": "summarization",
    "status": "canonical",
    "derived_from": None,
    "template_path": "prompt_assets/prompts/summarize/concise_v1.yaml",
    "input_schema": "summarize_input@1",
    "output_schema": "summarize_output@1",
    "tags": ["summarization", "reusable", "canonical"],
}
pprint(prompt_asset_manifest)


{'derived_from': None,
 'input_schema': 'summarize_input@1',
 'owner': 'shared',
 'output_schema': 'summarize_output@1',
 'ref': 'shared.summarize.concise@1',
 'status': 'canonical',
 'tags': ['summarization', 'reusable', 'canonical'],
 'task_family': 'summarization',
 'template_path': 'prompt_assets/prompts/summarize/concise_v1.yaml'}


In [4]:
derived_prompt_asset = {
    "ref": "research_v3.summarize.concise_for_contracts@1",
    "owner": "research_v3",
    "status": "project_candidate",
    "derived_from": "shared.summarize.concise@1",
    "template_path": "prompt_assets/prompts/research_v3/concise_for_contracts_v1.yaml",
    "tags": ["summarization", "contracts", "candidate"],
}
pprint(derived_prompt_asset)


{'derived_from': 'shared.summarize.concise@1',
 'owner': 'research_v3',
 'ref': 'research_v3.summarize.concise_for_contracts@1',
 'status': 'project_candidate',
 'tags': ['summarization', 'contracts', 'candidate'],
 'template_path': 'prompt_assets/prompts/research_v3/concise_for_contracts_v1.yaml'}


## Proposed Storage Layout

The key design choice is explicit identity and lineage, not symlink tricks.


In [5]:
storage_layout_options = [
    {
        "option": "shared prompt asset repo/package",
        "recommended": True,
        "reason": "gives one canonical place for reusable prompt assets and metadata",
    },
    {
        "option": "project-local overrides via shadowing",
        "recommended": False,
        "reason": "creates ambiguity for coding agents about which prompt actually ran",
    },
    {
        "option": "symlink-based reuse",
        "recommended": False,
        "reason": "file-level indirection is brittle and not provenance-friendly",
    },
    {
        "option": "derived assets with explicit lineage",
        "recommended": True,
        "reason": "preserves reuse while keeping source of truth explicit",
    },
]
pprint(storage_layout_options)


[{'option': 'shared prompt asset repo/package',
  'recommended': True,
  'reason': 'gives one canonical place for reusable prompt assets and metadata'},
 {'option': 'project-local overrides via shadowing',
  'recommended': False,
  'reason': 'creates ambiguity for coding agents about which prompt actually ran'},
 {'option': 'symlink-based reuse',
  'recommended': False,
  'reason': 'file-level indirection is brittle and not provenance-friendly'},
 {'option': 'derived assets with explicit lineage',
  'recommended': True,
  'reason': 'preserves reuse while keeping source of truth explicit'}]


## Resolver Contract Sketch

This is intentionally code-shaped but not yet implemented. The key rule is deterministic resolution from an explicit ref or an explicit path.


In [ ]:
# Target shape:
#
# from prompt_assets import resolve_prompt_asset
# from llm_client import render_prompt
#
# asset = resolve_prompt_asset("shared.summarize.concise@1")
# messages = render_prompt(asset.template_path, topic="...", style="...")
#
# Required resolver behavior:
# 1. explicit ref -> one manifest
# 2. explicit path -> one template file
# 3. missing ref -> fail loudly
# 4. ambiguous ref -> fail loudly
# 5. no hidden project-local shadowing in the default path
#
# Nice-to-have but not first slice:
# - tag search
# - prompt discovery UI
# - promotion workflow helpers


## Promotion Flow

The clean path is local experimentation first, explicit promotion second.


In [ ]:
# Suggested workflow:
#
# 1. start with a local YAML prompt file in a project repo
# 2. evaluate variants with prompt_eval
# 3. once a prompt is worth reusing, create a prompt asset manifest
# 4. assign a stable ref and move it into the shared prompt asset layer
# 5. if a project customizes it, create a new derived asset instead of overriding
#
# Pseudocode:
#
# candidate_ref = promote_prompt(
#     source_path="research_v3/prompts/summarize/concise.yaml",
#     new_ref="shared.summarize.concise@1",
#     metadata={"task_family": "summarization", "status": "canonical"},
# )
#
# derived_ref = derive_prompt(
#     parent_ref="shared.summarize.concise@1",
#     new_ref="research_v3.summarize.concise_for_contracts@1",
#     metadata={"status": "project_candidate", "domain": "contracts"},
# )


## Migration Phases

These are the thin slices for Phase 1 itself.


In [6]:
phase_1_breakdown = [
    {
        "name": "Phase 1a: Manifest schema",
        "goal": "define the prompt asset manifest and directory layout.",
        "done_when": [
            "one manifest schema exists",
            "ref/version/owner/derived_from/template_path are required or clearly optional",
            "at least one canonical example asset is checked in",
        ],
    },
    {
        "name": "Phase 1b: Resolver slice",
        "goal": "resolve explicit prompt refs deterministically.",
        "done_when": [
            "resolve_prompt_asset(ref) returns one manifest",
            "ambiguous or missing refs fail loudly",
            "llm_client can render from the resolved template path",
        ],
    },
    {
        "name": "Phase 1c: prompt_eval integration",
        "goal": "let prompt_eval variants be asset-first instead of inline-message-first.",
        "done_when": [
            "PromptVariant can be created from prompt_ref plus render context",
            "observability records prompt asset identity directly",
            "inline messages remain compatibility only",
        ],
    },
    {
        "name": "Phase 1d: Promotion workflow",
        "goal": "promote useful prompts from project-local experiments into shared assets.",
        "done_when": [
            "one documented promotion path exists",
            "derived assets record lineage explicitly",
            "no runtime shadowing is required",
        ],
    },
    {
        "name": "Phase 1e: Tighten policy",
        "goal": "reduce reliance on inline messages for reusable prompts.",
        "done_when": [
            "high-value reusable prompts use prompt_ref by default",
            "prompt asset refs appear in observability queries regularly",
            "inline messages are clearly marked as compatibility debt",
        ],
    },
]

for phase in phase_1_breakdown:
    print(phase["name"])
    print(f"  Goal: {phase['goal']}")
    print("  Done when:")
    for item in phase["done_when"]:
        print(f"    - {item}")
    print()


Phase 1a: Manifest schema
  Goal: define the prompt asset manifest and directory layout.
  Done when:
    - one manifest schema exists
    - ref/version/owner/derived_from/template_path are required or clearly optional
    - at least one canonical example asset is checked in

Phase 1b: Resolver slice
  Goal: resolve explicit prompt refs deterministically.
  Done when:
    - resolve_prompt_asset(ref) returns one manifest
    - ambiguous or missing refs fail loudly
    - llm_client can render from the resolved template path

Phase 1c: prompt_eval integration
  Goal: let prompt_eval variants be asset-first instead of inline-message-first.
  Done when:
    - PromptVariant can be created from prompt_ref plus render context
    - observability records prompt asset identity directly
    - inline messages remain compatibility only

Phase 1d: Promotion workflow
  Goal: promote useful prompts from project-local experiments into shared assets.
  Done when:
    - one documented promotion path exis

## Definition Of Done For Phase 1

This is the standard to hold the implementation to.


In [ ]:
phase_1_definition_of_done = {
    "must_be_true": [
        "a reusable prompt has one explicit ref and one explicit manifest",
        "derived prompts declare lineage explicitly",
        "llm_client can resolve and render an explicit prompt ref deterministically",
        "prompt_eval can evaluate prompt assets without relying on hidden override order",
        "observability records prompt asset identity in a query-friendly way",
    ],
    "must_not_be_true": [
        "runtime prompt selection depends on project-local shadowing",
        "symlink tricks are required to understand lineage",
        "a coding agent cannot tell which prompt actually ran",
    ],
}
pprint(phase_1_definition_of_done)


## Recommended Next Concrete Slice

The highest-leverage next step is not a full prompt platform. It is a minimal manifest + resolver slice with one real prompt asset and one real `prompt_eval` integration path.


In [ ]:
next_slice = {
    "implement_first": [
        "create prompt_assets manifest schema",
        "check in one canonical shared prompt asset",
        "add resolve_prompt_asset(ref)",
        "add one prompt_eval helper that builds a PromptVariant from prompt_ref + context",
    ],
    "do_not_build_yet": [
        "global prompt search UI",
        "tag-based recommender",
        "automatic prompt promotion pipeline",
        "project-local override system",
    ],
}
pprint(next_slice)
